# 02 — Model Experiments, Validation & Dual-Track Benchmarks
## EPFL CS-433 Machine Learning (Fall 2026) — Project 1 (`_MICHD`)

**Team:** Zhihao Chen, Zhiqian Zhou, Guanyu Chen  
- **Official Screening Ensemble (Clean SOTA, Top 14%):**  
  5-Fold CV F1 = **0.4368 ± 0.0026** | Held-Out F1 = **0.4340** | AIcrowd F1 = **0.447** (Accuracy = **87.7%**)
- **Audited Competition Variant (Leaky Benchmark, Top 13%):**  
  5-Fold CV F1 = **0.4464 ± 0.0032** | Held-Out F1 = **0.4445** | AIcrowd F1 = **0.457** (Accuracy = **88.3%**)

### Validation Protocol
To prevent data contamination, all evaluation follows a strict split:
* **20% Stratified Hold-Out Split** (65,627 rows, seed 2026) reserved strictly for final validation.
* **80% Development Set** (262,508 rows) used for 5-fold stratified cross-validation and hyperparameter selection.
* All preprocessing statistics and encodings are estimated strictly inside training folds.


## 1. Setup & Environment
We import project modules and load the BRFSS 2015 dataset from disk cache.
All models are implemented strictly in pure NumPy following OOP principles.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt

# Ensure repository root is on sys.path
REPO_ROOT = os.path.abspath('..') if os.path.exists('../implementations.py') else os.path.abspath('.')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import preprocessing as prep
import cross_validation as cv
from models import GradientBoostedTrees, MLPEnsemble, CardiacEnsemble
from experiments.plots import (
    load_table,
    show_table,
    plot_model_comparison,
    plot_f1_vs_rate,
    plot_leakage,
    plot_importance,
)

# Load BRFSS 2015 dataset from disk cache
data_dir = os.path.join(REPO_ROOT, 'data', 'dataset')
x_raw, _, y_raw, _, _ = prep.load_data_cached(data_dir)
feature_names = prep.load_feature_names(data_dir)

# 80/20 Stratified Holdout Isolation
dev_idx, holdout_idx = cv.holdout_split(y_raw, frac=0.20, seed=2026)
print(f'Dataset loaded: {x_raw.shape[0]:,} rows x {x_raw.shape[1]} features')
print(f'Development set: {len(dev_idx):,} rows | Held-out set: {len(holdout_idx):,} rows')


## 2. Target Distribution & Imbalance
- **Prevalence:** Only **8.83%** of respondents experienced coronary heart disease or myocardial infarction (10:1 skew).
- **The Accuracy Fallacy:** A trivial majority predictor (predicting -1 for everyone) achieves **91.17% accuracy**, but **F1 = 0.0000**.
- **Metric:** Models are ranked strictly by positive-class F1 score.


In [ ]:
y_dev = y_raw[dev_idx]
n_pos = int(np.sum(y_dev == 1))
n_neg = int(np.sum(y_dev == -1))
prev = n_pos / len(y_dev)

fig, ax = plt.subplots(figsize=(5.5, 2.8), dpi=150)
bars = ax.bar(['Negative (-1)', 'Positive (+1)'], [n_neg, n_pos], color=['#3b82f6', '#ef4444'], width=0.45, edgecolor='black')
for b, count in zip(bars, [n_neg, n_pos]):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 3000, f'{count:,}\n({count/len(y_dev)*100:.1f}%)', ha='center', fontsize=8)
ax.set_ylim(0, 270000)
ax.set_ylabel('Respondent Count', fontsize=9)
ax.set_title(f'BRFSS 2015 Target Imbalance (Prevalence = {prev*100:.2f}%)', fontsize=10, fontweight='bold')
ax.grid(axis='y', linestyle=':', alpha=0.5)
fig.tight_layout()
plt.show()


## 3. Linear Baselines & The Threshold Leap (0.23 -> 0.43)
- **Standard Baseline Performance:** Under the default probability cutoff of 0.5, logistic regression yields an F1 of only **0.2302** due to extreme class imbalance.
- **The Percentile Rule:** By selecting the top 14% highest-risk individuals, F1 immediately jumps to **0.4302 (+0.200 gain)** without altering any model weights.
- **Linear Ceiling:** All six course baseline methods (`Least Squares`, `Ridge`, `MSE GD`, `Logistic GD`, `Reg. Logistic`) plateau around **0.4216 – 0.4302**, establishing the linear benchmark ceiling.


In [ ]:
# Grounding Report Table I: Full Baseline Benchmark
show_table('baselines_full', ['model', 'features', 'cv_f1_mean', 'cv_f1_std', 'holdout_f1'])


## 4. Live Demo: Pure-NumPy OOP Model Fitting
We demonstrate the refactored, pure-NumPy OOP implementations on a representative development subsample (3,000 rows):
1. **`GradientBoostedTrees`**: 64-bin histogram trees, Newton leaves ($\lambda=5$), early stopping, depth 5.
2. **`MLPEnsemble`**: 64 ReLU hidden units, Adam optimizer, weight decay, multi-seed averaging.
3. **`CardiacEnsemble`**: 50/50 blend of tree logits and neural network logits.


In [ ]:
# Subsample for live fast demonstration (3,000 rows in <3 seconds)
rng = np.random.default_rng(42)
sub_idx = rng.choice(dev_idx, size=3000, replace=False)
x_sub = x_raw[sub_idx]
y_sub = y_raw[sub_idx]

# Clean features and transform with preprocessor
cleaner = prep.BRFSSCleaner()
xc_sub, names_sub = cleaner.clean(x_sub, feature_names)

preprocessor = prep.TabularPreprocessor()
tx_sub = preprocessor.fit_transform(xc_sub, names_sub)

# 1. Fit OOP Histogram Gradient Boosted Trees
t0 = time.time()
gbm = GradientBoostedTrees(n_trees=20, max_depth=4, lr=0.08, min_leaf=30, patience=5)
gbm.fit(tx_sub, y_sub)
gbm_scores = gbm.predict_logits(tx_sub)
print(f'[1/2] GradientBoostedTrees fitted in {time.time()-t0:.2f}s ({len(gbm.trees_)} trees)')

# 2. Fit OOP Multi-Seed Neural Network Ensemble
t0 = time.time()
mlp = MLPEnsemble(seed=0, n_seeds=3, hidden=(32,), max_epochs=8, lr=0.005, batch_size=256)
mlp.fit(tx_sub, y_sub)
mlp_scores = mlp.predict_logits(tx_sub)
print(f'[2/2] MLPEnsemble fitted in {time.time()-t0:.2f}s (3 seeds)')

# 3. Blend scores
blend_scores = 0.5 * gbm_scores + 0.5 * mlp_scores
print(f'Blend logit range: [{blend_scores.min():.2f}, {blend_scores.max():.2f}]')


## 5. Model Comparison (Report Table I)
Full-dataset benchmarking across all model milestones on the 5-fold stratified development set (262,508 rows) and 20% held-out set (65,627 rows).
- Non-linear models surpass the linear ceiling: **Histogram GBM** reaches **0.4354** CV F1, and **10-seed MLP** reaches **0.4334**.
- Blending both into **GBM + MLP Ensemble** achieves the best clean performance: **0.4368 ± 0.0026** CV F1 and **0.4340** held-out F1, scoring **0.447** on AIcrowd.


In [ ]:
comp_rows = load_table('model_comparison')
show_table('model_comparison', ['model', 'family', 'cv_f1_mean', 'cv_f1_std', 'holdout_f1'])
fig = plot_model_comparison(comp_rows)
plt.show()


## 6. Decision Rule Optimization (Report Figure 1)
Instead of an arbitrary 0.5 probability cutoff, models predict positive for the top-scored fraction of respondents.
- **Stability:** As shown below, positive-class F1 forms a broad, stable plateau between **12% and 16%**.
- **Chosen Cutoffs:**
  - **Clean Model:** Top **14.0%** (balancing precision ~0.354 and recall ~0.561).
  - **Leaky Model:** Top **12.8% ~ 13.0%** (maximizing F1 to 0.4464).


In [ ]:
rate_rows = load_table('f1_vs_rate')
fig = plot_f1_vs_rate(rate_rows)
plt.show()


## 7. Target Leakage Clinical Audit (Report Table II & Section V)
Seven BRFSS survey questions inquire about care *following* a coronary event or stroke:
- Cardiac and stroke rehabilitation: `HAREHAB1`, `STREHAB1`
- Secondary prevention aspirin regimen: `CVDASPRN`, `RDUCHART`, `RDUCSTRK`, `RLIVPAIN`, `ASPUNSAF`

**The Reverse Causality Trap:**  
All 615 development respondents (and 158/158 held-out respondents) answering `HAREHAB1` are confirmed positive (100% precision). Incorporating these 7 features inflates AIcrowd F1 to **0.457** (+0.010 gain).  
However, undiagnosed patients in pre-event screening cannot have undergone cardiac rehab. Exploiting these questions is clinically invalid.  
Therefore, our **Official Screening Model strictly excludes all 7 leaky features**, while documenting the leaky model as a competition benchmark (`python run.py`).


In [ ]:
summary_rows = load_table('final_summary')
print('=== Report Table II: Dual-Track Comparison ===')
show_table('final_summary', ['version', 'cv_f1_mean', 'cv_f1_std', 'holdout_f1', 'aicrowd_f1', 'aicrowd_accuracy'])

leak_rows = load_table('leakage_ablation')
fig = plot_leakage(leak_rows)
plt.show()

print('\n=== Single-Column Leakage Ablation ===')
show_table('leaky_columns', ['column', 'lr_clean_plus_col_nested', 'delta_vs_clean', 'won_vs_clean', 'delta_vs_leaky_all'])


## 8. Feature Importance & Clinical Risk Rules (Report Section IV)
- **Permutation Importance:** Evaluating the drop in test AUC when individual features are permuted reveals the strongest drivers: Age (`_AGE80`, 0.035 drop), General Health (`GENHLTH`, 0.021), Sex (`SEX`, 0.018), Cholesterol (`TOLDHI2`, 0.009), and prior Stroke (`CVDSTRK3`, 0.008).
- **Clinical Rules:** High-risk combinations multiply disease risk: a prior stroke carries a **4.24-fold risk lift**, and $\ge 5$ chronic conditions carry a **3.72-fold lift**.


In [ ]:
imp_rows = load_table('feature_importance')
fig = plot_importance(imp_rows, top=10)
plt.show()

print('\n=== Verified Clinical Risk Rules (Dev Set) ===')
show_table('rules', ['rule', 'support', 'precision', 'recall', 'lift'])


## 9. Submission File Verification
We verify that the generated submission files strictly adhere to competition requirements:
1. Exactly 109,379 prediction rows with columns `Id,Prediction`.
2. Positive counts match the calibrated cutoff: **14,876** for Clean SOTA (0.447) and **14,001** for Leaky Champion (0.457).


In [ ]:
def verify_submission(filename, expected_positives):
    path = os.path.join(REPO_ROOT, 'submissions', filename)
    if not os.path.exists(path):
        print(f'Skipping {filename}: not yet generated.')
        return
    with open(path, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]
    header = lines[0]
    preds = [int(line.split(',')[1]) for line in lines[1:]]
    n_pos = sum(1 for p in preds if p == 1)
    n_total = len(preds)
    assert header == 'Id,Prediction', f'Invalid header: {header}'
    assert n_total == 109379, f'Invalid row count: {n_total}'
    assert n_pos == expected_positives, f'Positives mismatch: {n_pos} != {expected_positives}'
    print(f'Verified {filename}: {n_total:,} rows, {n_pos:,} positives ({n_pos/n_total*100:.2f}%) - OK')

verify_submission('submission_clean_sota.csv', 14876)
verify_submission('submission_with_leakage.csv', 14001)
